# 03 — Momentum strategy workshop

This notebook builds a daily momentum strategy in four steps:

1. define a moving-average signal;
2. backtest it with correct timing and transaction costs;
3. study parameter sensitivity using training and test periods;
4. extend the same idea to EWMA and cross-sectional momentum.

The code uses Alpaca daily bars. The `.env` file should contain `API_KEY` and `SECRET_KEY`.


## 0. Setup

Required packages: `pandas`, `numpy`, `plotly`, `alpaca-py`, and `python-dotenv`.

The setup cell also defines the plotting and performance functions used later.


In [50]:
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from alpaca.data.enums import Adjustment, DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from dotenv import find_dotenv, load_dotenv
from IPython.display import display
from plotly.subplots import make_subplots

ENV_PATH = find_dotenv(usecwd=True)
if ENV_PATH:
    load_dotenv(ENV_PATH)

API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")

if not API_KEY or not SECRET_KEY:
    raise ValueError(
        "Missing API_KEY or SECRET_KEY. Put them in the .env file used by the TigerQuant setup."
    )

client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

BACKGROUND = "#EEEEEE"
TEXT = "#111111"
MUTED_TEXT = "#555555"
GRIDLINE = "#D2D2D2"
PRICE = "#2F4858"
ACCENT = "#008C95"
SECONDARY_ACCENT = "#C56A13"
FONT_FAMILY = "Times New Roman"


def style_chart(
    figure: go.Figure,
    *,
    title: str | None = None,
    source_text: str | None = None,
    width: int = 1250,
    height: int = 520,
    legend: dict[str, Any] | None = None,
) -> go.Figure:
    legend_config = {
        "orientation": "h",
        "x": 0.0,
        "y": 1.04,
        "xanchor": "left",
        "yanchor": "bottom",
        "font": {"family": FONT_FAMILY, "size": 16},
        "bgcolor": "rgba(0,0,0,0)",
    }
    if legend:
        legend_config.update(legend)

    figure.update_layout(
        width=width,
        height=height,
        paper_bgcolor=BACKGROUND,
        plot_bgcolor=BACKGROUND,
        margin={"l": 75, "r": 30, "t": 90, "b": 65},
        font={"family": FONT_FAMILY, "size": 17, "color": TEXT},
        legend=legend_config,
        hovermode="x unified",
        title={
            "text": title or "",
            "x": 0.0,
            "xanchor": "left",
            "font": {"family": FONT_FAMILY, "size": 26, "color": TEXT},
        },
    )

    if source_text:
        figure.add_annotation(
            text=source_text,
            x=1.0,
            y=1.13,
            xref="paper",
            yref="paper",
            showarrow=False,
            xanchor="right",
            font={"family": FONT_FAMILY, "size": 14, "color": MUTED_TEXT},
        )

    return figure


def label_axes(figure, *, x_title, y_title, y_tick_format=None):
    figure.update_xaxes(
        title_text=x_title,
        showgrid=False,
        showline=True,
        linecolor="#777777",
        ticks="outside",
        zeroline=False,
    )
    figure.update_yaxes(
        title_text=y_title,
        showgrid=True,
        gridcolor=GRIDLINE,
        showline=True,
        linecolor="#777777",
        ticks="outside",
        zeroline=False,
        tickformat=y_tick_format,
    )
    return figure


def fetch_bars(symbol, timeframe, start, end, *, adjustment=Adjustment.ALL):
    request = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=timeframe,
        start=start,
        end=end,
        adjustment=adjustment,
        feed=DataFeed.IEX,
        limit=10_000,
    )

    raw = client.get_stock_bars(request).df

    if raw.empty:
        raise ValueError(f"Alpaca returned no bars for {symbol}")

    frame = raw.reset_index().copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True)

    frame = (
        frame.sort_values("timestamp")
        .drop_duplicates("timestamp")
        .set_index("timestamp")
    )

    return frame


def max_drawdown(returns):
    equity = (1 + returns.fillna(0.0)).cumprod()
    return (equity / equity.cummax() - 1).min()


def summary_from_returns(
    returns,
    *,
    periods_per_year,
    position=None,
    turnover=None,
):
    returns = returns.dropna()

    if len(returns) == 0:
        raise ValueError("No returns were supplied.")

    equity = (1 + returns).cumprod()
    years = max(len(returns) / periods_per_year, 1 / periods_per_year)
    vol = returns.std(ddof=1)

    summary = {
        "Total return": equity.iloc[-1] - 1,
        "Annualized return": equity.iloc[-1] ** (1 / years) - 1,
        "Annualized volatility": vol * np.sqrt(periods_per_year),
        "Sharpe ratio": (
            np.nan
            if vol == 0
            else returns.mean() / vol * np.sqrt(periods_per_year)
        ),
        "Maximum drawdown": max_drawdown(returns),
    }

    if position is not None:
        position = position.reindex(returns.index)
        summary["Entries"] = int(
            (position.gt(0) & position.shift(1).fillna(0).eq(0)).sum()
        )
        summary["Time in market"] = position.mean()

    if turnover is not None:
        turnover = turnover.reindex(returns.index)
        summary["Turnover"] = turnover.sum()

    return summary


def sharpe_ratio(returns, periods_per_year=252):
    returns = returns.dropna()
    vol = returns.std(ddof=1)
    return (
        np.nan
        if len(returns) == 0 or vol == 0
        else returns.mean() / vol * np.sqrt(periods_per_year)
    )


print(
    f"Loaded environment: "
    f"{Path(ENV_PATH).name if ENV_PATH else 'environment variables'}"
)


Loaded environment: .env


## 1. Daily SPY data

We use adjusted daily bars. The sample begins in 2020 and ends at the most recent completed trading day.

The continuity check prevents a long missing-data interval from being treated as one daily return.


In [51]:
SYMBOL = "SPY"
DAILY_START = datetime(2020, 1, 1, tzinfo=timezone.utc)
DAILY_END = (
    datetime.now(timezone.utc)
    .replace(hour=0, minute=0, second=0, microsecond=0)
    - timedelta(days=1)
)

prices = fetch_bars(
    SYMBOL,
    TimeFrame.Day,
    DAILY_START,
    DAILY_END,
)[["open", "high", "low", "close", "volume"]].copy()

day_gaps = prices.index.to_series().diff().dt.days
large_gaps = day_gaps[day_gaps > 10]

if not large_gaps.empty:
    last_gap_end = large_gaps.index[-1]
    print(
        f"Large data gap found before {last_gap_end.date()}. "
        "Using the continuous segment after the gap."
    )
    prices = prices.loc[last_gap_end:].copy()

assert prices.index.is_monotonic_increasing
assert prices["close"].gt(0).all()

display(
    pd.DataFrame(
        {
            "value": [
                prices.index.min().date(),
                prices.index.max().date(),
                len(prices),
            ]
        },
        index=["first session", "last session", "rows"],
    )
)

display(prices.head())


,value
first session,2020-07-27
last session,2026-09-21
rows,1546


,open,high,low,close,volume
timestamp,,,,,
2020-07-27 04:00:00+00:00,295.44,297.06,294.70,296.94,647693.0
2020-07-28 04:00:00+00:00,296.31,297.29,294.73,295.09,521475.0
2020-07-29 04:00:00+00:00,295.89,299.19,295.89,298.69,600377.0
2020-07-30 04:00:00+00:00,295.52,297.97,293.62,297.58,612074.0
2020-07-31 04:00:00+00:00,299.37,300.01,295.18,299.95,1135945.0


In [52]:
fig = go.Figure(
    go.Scatter(
        x=prices.index,
        y=prices["close"],
        name="SPY",
        line={"color": PRICE, "width": 2.2},
    )
)

style_chart(
    fig,
    title="SPY adjusted close",
    source_text="Alpaca IEX daily bars",
)
label_axes(fig, x_title="Date", y_title="Price ($)")
fig.show()


## 2. Simple moving averages

For closing price $P_t$, the $n$-day simple moving average is

$$
\operatorname{SMA}_n(t)
=
\frac{1}{n}
\sum_{j=0}^{n-1} P_{t-j}.
$$

The fast average uses a shorter window and therefore reacts more quickly to recent price changes. The slow average changes more gradually.

We begin with 20 and 100 trading days.

### Exercise

Complete the two rolling means.

```python
strategy["fast_ma"] = strategy["close"].________(FAST_WINDOW).mean()
strategy["slow_ma"] = strategy["close"].________(SLOW_WINDOW).mean()
```

<details>
<summary>Hint</summary>

`rolling(window)` constructs a trailing window.

</details>

<details>
<summary>Solution</summary>

```python
strategy["fast_ma"] = strategy["close"].rolling(FAST_WINDOW).mean()
strategy["slow_ma"] = strategy["close"].rolling(SLOW_WINDOW).mean()
```

</details>

### Check

Which pair should react more quickly to a change in price: 10/200 or 50/100?

<details>
<summary>Solution</summary>

The 10/200 pair. Its fast average uses only 10 observations, so the short-horizon component responds more quickly.

</details>


In [53]:
# Exercise cell. Uncomment the last two lines and complete them.

FAST_WINDOW = 20
SLOW_WINDOW = 100

practice = prices.copy()

# practice["fast_ma"] = practice["close"].________(FAST_WINDOW).mean()
# practice["slow_ma"] = practice["close"].________(SLOW_WINDOW).mean()


In [54]:
FAST_WINDOW = 20
SLOW_WINDOW = 100

strategy = prices.copy()

strategy["fast_ma"] = (
    strategy["close"]
    .rolling(FAST_WINDOW, min_periods=FAST_WINDOW)
    .mean()
)

strategy["slow_ma"] = (
    strategy["close"]
    .rolling(SLOW_WINDOW, min_periods=SLOW_WINDOW)
    .mean()
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=strategy.index,
        y=strategy["close"],
        name="SPY",
        line={"color": PRICE, "width": 1.5},
    )
)

fig.add_trace(
    go.Scatter(
        x=strategy.index,
        y=strategy["fast_ma"],
        name=f"{FAST_WINDOW}-day SMA",
        line={"color": ACCENT, "width": 2.2},
    )
)

fig.add_trace(
    go.Scatter(
        x=strategy.index,
        y=strategy["slow_ma"],
        name=f"{SLOW_WINDOW}-day SMA",
        line={"color": SECONDARY_ACCENT, "width": 2.2},
    )
)

style_chart(
    fig,
    title="Fast and slow simple moving averages",
    source_text="Daily SPY",
)
label_axes(fig, x_title="Date", y_title="Price ($)")
fig.show()


## 3. Define the signal

The strategy is long-only. Define

$$
s_t
=
\begin{cases}
1, & \text{if } \operatorname{SMA}_{20}(t) > \operatorname{SMA}_{100}(t),\\
0, & \text{otherwise.}
\end{cases}
$$

The signal is a desired position. A trade occurs only when the signal changes.

### Exercise - fill in the blank

```python
strategy["signal"] = np.where(
    ready & strategy["fast_ma"].____(strategy["slow_ma"]),
    1.0,
    0.0,
)
```

<details>
<summary>Hint</summary>

`.gt()` compares whether the left series is greater than the right series.

</details>

<details>
<summary>Solution</summary>

```python
strategy["signal"] = np.where(
    ready & strategy["fast_ma"].gt(strategy["slow_ma"]),
    1.0,
    0.0,
)
```

</details>

### Check

Suppose the signal equals 1 for 40 consecutive trading days. How many entries occur during that interval?

<details>
<summary>Solution</summary>

One entry occurs when the signal changes from 0 to 1. The strategy then remains long until the signal changes again.

</details>


In [55]:
# Exercise cell.

# ready = practice[["fast_ma", "slow_ma"]].notna().all(axis=1)
# practice["signal"] = np.where(
#     ready & practice["fast_ma"].__(practice["slow_ma"]),
#     1.0,
#     0.0,
# )


In [56]:
ready = strategy[["fast_ma", "slow_ma"]].notna().all(axis=1)

strategy["signal"] = np.where(
    ready & strategy["fast_ma"].gt(strategy["slow_ma"]),
    1.0,
    0.0,
)

strategy["trade"] = (
    strategy["signal"]
    .diff()
    .fillna(strategy["signal"])
)

display(
    strategy[
        ["close", "fast_ma", "slow_ma", "signal", "trade"]
    ]
    .dropna()
    .head()
)


,close,fast_ma,slow_ma,signal,trade
timestamp,,,,,
2020-12-15 05:00:00+00:00,340.84,335.794,317.0017,1.0,1.0
2020-12-16 05:00:00+00:00,341.33,336.229,317.4456,1.0,0.0
2020-12-17 05:00:00+00:00,343.25,336.964,317.9272,1.0,0.0
2020-12-18 05:00:00+00:00,341.87,337.556,318.3590,1.0,0.0
2020-12-21 05:00:00+00:00,340.71,338.204,318.7903,1.0,0.0


## 4. Backtest timing

The closing price at time $t$ is used to calculate the signal at time $t$. That signal cannot earn the return that already occurred during the same period.

The backtest therefore uses

$$
\text{position}_t = s_{t-1}.
$$

### Check

Which pandas operation implements this one-period delay?

<details>
<summary>Hint</summary>

The signal must move forward by one row.

</details>

<details>
<summary>Solution</summary>

```python
position = signal.shift(1)
```

</details>


In [57]:
COST_BPS = 2.0

strategy["asset_return"] = (
    strategy["close"]
    .pct_change(fill_method=None)
    .fillna(0.0)
)

# Incorrect timing for comparison.
strategy["same_day_position"] = strategy["signal"]
strategy["same_day_return"] = (
    strategy["same_day_position"]
    * strategy["asset_return"]
)

# Correct timing.
strategy["position"] = (
    strategy["signal"]
    .shift(1)
    .fillna(0.0)
)

strategy["position_change"] = (
    strategy["position"]
    .diff()
    .fillna(strategy["position"])
)

strategy["turnover"] = strategy["position_change"].abs()

strategy["gross_return"] = (
    strategy["position"]
    * strategy["asset_return"]
)

strategy["transaction_cost"] = (
    strategy["turnover"]
    * COST_BPS
    / 10_000
)

strategy["net_return"] = (
    strategy["gross_return"]
    - strategy["transaction_cost"]
)

VALID_START = strategy["slow_ma"].first_valid_index()

timing_comparison = pd.DataFrame(
    {
        "same-day signal": [
            (1 + strategy.loc[VALID_START:, "same_day_return"]).prod() - 1
        ],
        "shifted signal": [
            (1 + strategy.loc[VALID_START:, "gross_return"]).prod() - 1
        ],
    },
    index=["Total return"],
)

display(timing_comparison.round(4))


,same-day signal,shifted signal
Total return,0.9254,0.8352


## 5. Transaction costs and performance

A change in position creates turnover. With a fixed cost of $c$ basis points,

$$
\text{cost}_t
=
\left|\text{position}_t-\text{position}_{t-1}\right|
\frac{c}{10{,}000}.
$$

The net strategy return is

$$
r^{\text{net}}_t
=
\text{position}_t r_t-\text{cost}_t.
$$

We compare total return, annualized volatility, Sharpe ratio, maximum drawdown, turnover, and time in market.


In [58]:
evaluation = strategy.loc[VALID_START:].copy()

equity = pd.DataFrame(
    {
        "Buy and hold": (1 + evaluation["asset_return"]).cumprod(),
        "Momentum gross": (1 + evaluation["gross_return"]).cumprod(),
        "Momentum net": (1 + evaluation["net_return"]).cumprod(),
    }
)

fig = go.Figure()

for name, color in zip(
    equity.columns,
    [PRICE, ACCENT, SECONDARY_ACCENT],
):
    fig.add_trace(
        go.Scatter(
            x=equity.index,
            y=equity[name],
            name=name,
            line={"color": color, "width": 2.2},
        )
    )

style_chart(
    fig,
    title="20/100 SMA momentum backtest",
    source_text=f"{COST_BPS:.0f} bps per position change",
)
label_axes(fig, x_title="Date", y_title="Growth of $1")
fig.show()

metrics = pd.DataFrame(
    {
        "Buy and hold": summary_from_returns(
            evaluation["asset_return"],
            periods_per_year=252,
        ),
        "Momentum gross": summary_from_returns(
            evaluation["gross_return"],
            periods_per_year=252,
            position=evaluation["position"],
            turnover=evaluation["turnover"],
        ),
        "Momentum net": summary_from_returns(
            evaluation["net_return"],
            periods_per_year=252,
            position=evaluation["position"],
            turnover=evaluation["turnover"],
        ),
    }
)

display(metrics.round(4))


,Buy and hold,Momentum gross,Momentum net
Total return,1.2999,0.8352,0.8304
Annualized return,0.1561,0.1115,0.1110
Annualized volatility,0.1641,0.1173,0.1173
Sharpe ratio,0.9662,0.9605,0.9567
Maximum drawdown,-0.2451,-0.1922,-0.1928
Entries,NaN,7.0000,7.0000
Time in market,NaN,0.7892,0.7892
Turnover,NaN,13.0000,13.0000


## 6. Parameter choice

The parameter range should be chosen before inspecting the backtest results.

For daily data, use:

- fast window: 5 to 50 trading days in 5-day increments;
- slow window: 60 to 300 trading days in 20-day increments.

These ranges represent different short- and long-horizon trends while preserving the interpretation that the fast average is shorter than the slow average.

The comparison also uses the same evaluation dates for every parameter pair. Otherwise, a longer slow window would be evaluated on a shorter sample.

### Check

Why should the parameter range be chosen before selecting the best Sharpe ratio?

<details>
<summary>Solution</summary>

The range should follow the intended trading horizon. If the range is chosen after seeing the results, the parameter search itself becomes part of the fitting process.

</details>


In [59]:
def run_sma_crossover(
    close,
    fast_window,
    slow_window,
    cost_bps=2.0,
):
    frame = pd.DataFrame({"close": close}).copy()

    frame["fast_ma"] = (
        frame["close"]
        .rolling(fast_window, min_periods=fast_window)
        .mean()
    )

    frame["slow_ma"] = (
        frame["close"]
        .rolling(slow_window, min_periods=slow_window)
        .mean()
    )

    ready = frame[["fast_ma", "slow_ma"]].notna().all(axis=1)

    frame["signal"] = np.where(
        ready & frame["fast_ma"].gt(frame["slow_ma"]),
        1.0,
        0.0,
    )

    frame["position"] = (
        frame["signal"]
        .shift(1)
        .fillna(0.0)
    )

    frame["turnover"] = (
        frame["position"]
        .diff()
        .fillna(frame["position"])
        .abs()
    )

    frame["asset_return"] = (
        frame["close"]
        .pct_change(fill_method=None)
        .fillna(0.0)
    )

    frame["gross_return"] = (
        frame["position"]
        * frame["asset_return"]
    )

    frame["net_return"] = (
        frame["gross_return"]
        - frame["turnover"] * cost_bps / 10_000
    )

    return frame


FAST_WINDOWS = list(range(5, 51, 5))
SLOW_WINDOWS = list(range(60, 301, 20))

COMMON_START = prices.index[SLOW_WINDOWS[-1]]

common_dates = prices.loc[COMMON_START:].index
split_row = int(len(common_dates) * 0.70)
SPLIT_DATE = common_dates[split_row]

print(f"Common evaluation start: {COMMON_START.date()}")
print(f"Test period begins:      {SPLIT_DATE.date()}")


Common evaluation start: 2021-10-04
Test period begins:      2025-03-26


## 7. Training and test periods

The first 70% of the common evaluation sample is the training period. The last 30% is the test period.

Parameters are compared using training data. The test period is then used to evaluate the selected rule on later observations.

### Exercise

For one parameter pair, which rows belong in the training sample?

```python
train = result.loc[result.index ______ SPLIT_DATE]
```

<details>
<summary>Hint</summary>

The training observations occur before the split date.

</details>

<details>
<summary>Solution</summary>

```python
train = result.loc[result.index < SPLIT_DATE]
```

</details>


In [60]:
grid_rows = []

for fast in FAST_WINDOWS:
    for slow in SLOW_WINDOWS:
        result = run_sma_crossover(
            prices["close"],
            fast,
            slow,
            COST_BPS,
        ).loc[COMMON_START:]

        train = result.loc[result.index < SPLIT_DATE]
        test = result.loc[result.index >= SPLIT_DATE]

        grid_rows.append(
            {
                "fast": fast,
                "slow": slow,
                "train_sharpe": sharpe_ratio(
                    train["net_return"],
                    periods_per_year=252,
                ),
                "test_sharpe": sharpe_ratio(
                    test["net_return"],
                    periods_per_year=252,
                ),
                "train_return": (
                    (1 + train["net_return"]).prod() - 1
                ),
                "test_return": (
                    (1 + test["net_return"]).prod() - 1
                ),
                "turnover": result["turnover"].sum(),
            }
        )

parameter_grid = pd.DataFrame(grid_rows)

display(
    parameter_grid
    .sort_values("train_sharpe", ascending=False)
    .head(12)
    .round(3)
)

selected = (
    parameter_grid
    .sort_values("train_sharpe", ascending=False)
    .iloc[0]
)

print(
    "Highest training Sharpe: "
    f"{int(selected.fast)}/{int(selected.slow)}"
)


,fast,slow,train_sharpe,test_sharpe,train_return,test_return,turnover
126,50,240,1.144,1.028,0.573,0.253,6.0
109,45,160,1.112,0.707,0.554,0.132,4.0
84,35,180,1.096,0.761,0.537,0.145,4.0
122,50,160,1.090,0.536,0.545,0.095,4.0
97,40,180,1.082,0.719,0.527,0.135,4.0
113,45,240,1.080,0.731,0.526,0.165,6.0
125,50,220,1.055,0.689,0.515,0.153,4.0
96,40,160,1.054,0.560,0.514,0.099,8.0
127,50,260,1.027,1.074,0.505,0.267,4.0
108,45,140,1.023,0.930,0.493,0.152,8.0


Highest training Sharpe: 50/240


In [61]:
train_heat = (
    parameter_grid
    .pivot(index="fast", columns="slow", values="train_sharpe")
    .reindex(index=FAST_WINDOWS, columns=SLOW_WINDOWS)
)

test_heat = (
    parameter_grid
    .pivot(index="fast", columns="slow", values="test_sharpe")
    .reindex(index=FAST_WINDOWS, columns=SLOW_WINDOWS)
)

z_values = np.concatenate(
    [
        train_heat.to_numpy().ravel(),
        test_heat.to_numpy().ravel(),
    ]
)

z_values = z_values[np.isfinite(z_values)]
z_bound = max(abs(z_values.min()), abs(z_values.max()))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Training Sharpe", "Test Sharpe"),
    horizontal_spacing=0.08,
)

for col, matrix in enumerate([train_heat, test_heat], start=1):
    fig.add_trace(
        go.Heatmap(
            x=matrix.columns,
            y=matrix.index,
            z=matrix.to_numpy(),
            colorscale="RdYlGn",
            zmin=-z_bound,
            zmax=z_bound,
            zmid=0,
            colorbar={"title": "Sharpe"} if col == 2 else None,
            showscale=(col == 2),
            hovertemplate=(
                "Slow=%{x}<br>"
                "Fast=%{y}<br>"
                "Sharpe=%{z:.2f}<extra></extra>"
            ),
        ),
        row=1,
        col=col,
    )

style_chart(
    fig,
    title="SMA parameter sensitivity",
    source_text=f"{COST_BPS:.0f} bps per position change",
    width=1400,
    height=620,
)

fig.update_xaxes(title_text="Slow window")
fig.update_yaxes(
    title_text="Fast window",
    autorange="reversed",
)

fig.show()


### Interpreting the sensitivity table

Parameter sensitivity shows how much the result changes when the moving-average windows change.

A stable area of similar values indicates that the result is not tied to one exact parameter pair. A single high value surrounded by materially different results indicates greater parameter sensitivity.

### Check

Suppose 20/100 has the highest training Sharpe, while 15/100, 20/120, and 25/100 all have much lower values. What does the table show?

<details>
<summary>Solution</summary>

The result is sensitive to the exact parameter choice. The 20/100 result should therefore be interpreted cautiously even if its training Sharpe is the largest.

</details>


In [62]:
selected_fast = int(selected["fast"])
selected_slow = int(selected["slow"])

selected_bt = run_sma_crossover(
    prices["close"],
    selected_fast,
    selected_slow,
    COST_BPS,
).loc[COMMON_START:]

selected_train = selected_bt.loc[
    selected_bt.index < SPLIT_DATE
]

selected_test = selected_bt.loc[
    selected_bt.index >= SPLIT_DATE
]

selected_metrics = pd.DataFrame(
    {
        "Training": summary_from_returns(
            selected_train["net_return"],
            periods_per_year=252,
            position=selected_train["position"],
            turnover=selected_train["turnover"],
        ),
        "Test": summary_from_returns(
            selected_test["net_return"],
            periods_per_year=252,
            position=selected_test["position"],
            turnover=selected_test["turnover"],
        ),
    }
)

display(selected_metrics.round(4))


,Training,Test
Total return,0.5729,0.2527
Annualized return,0.1398,0.1639
Annualized volatility,0.1209,0.1601
Sharpe ratio,1.1437,1.0279
Maximum drawdown,-0.1286,-0.1269
Entries,3.0000,2.0000
Time in market,0.7408,0.8957
Turnover,4.0000,2.0000


## 8. Exponentially weighted moving averages

A simple moving average assigns equal weight to the observations inside its window. An exponentially weighted moving average assigns larger weights to more recent observations.

With `adjust=False`, pandas uses the recursive form

$$
E_t
=
\alpha P_t
+
(1-\alpha)E_{t-1},
$$

where

$$
\alpha
=
\frac{2}{\text{span}+1}.
$$

For a span of 20,

$$
\alpha=\frac{2}{21}\approx 0.095.
$$

The current close therefore receives about 9.5% of the weight in the new EWMA value. Earlier observations remain in the average with geometrically decreasing weights.

A 20-day SMA and a 20-span EWMA are comparable horizons, but they are not the same weighting rule.

### Check

For the same nominal horizon, which average reacts more strongly to the most recent observation: SMA or EWMA?

<details>
<summary>Solution</summary>

EWMA. Its weights are largest for the most recent observations, while the SMA assigns equal weight to every observation inside the window.

</details>


### Exercise

Complete the EWMA calculation.

```python
strategy["fast_ewma"] = strategy["close"].ewm(
    span=FAST_WINDOW,
    adjust=____,
).mean()
```

<details>
<summary>Hint</summary>

Use the recursive definition shown above.

</details>

<details>
<summary>Solution</summary>

```python
strategy["fast_ewma"] = strategy["close"].ewm(
    span=FAST_WINDOW,
    adjust=False,
).mean()
```

</details>


In [63]:
# Exercise cell.

# strategy["fast_ewma"] = strategy["close"].ewm(
#     span=FAST_WINDOW,
#     adjust=____,
# ).mean()


In [64]:
strategy["fast_ewma"] = (
    strategy["close"]
    .ewm(
        span=FAST_WINDOW,
        adjust=False,
        min_periods=FAST_WINDOW,
    )
    .mean()
)

strategy["slow_ewma"] = (
    strategy["close"]
    .ewm(
        span=SLOW_WINDOW,
        adjust=False,
        min_periods=SLOW_WINDOW,
    )
    .mean()
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=strategy.index,
        y=strategy["fast_ma"],
        name=f"{FAST_WINDOW}-day SMA",
        line={"color": ACCENT, "width": 1.8},
    )
)

fig.add_trace(
    go.Scatter(
        x=strategy.index,
        y=strategy["fast_ewma"],
        name=f"{FAST_WINDOW}-span EWMA",
        line={"color": PRICE, "width": 1.8, "dash": "dash"},
    )
)

style_chart(
    fig,
    title="SMA and EWMA at the fast horizon",
    source_text="Daily SPY",
)
label_axes(fig, x_title="Date", y_title="Moving average")
fig.show()


In [65]:
def run_ewma_crossover(
    close,
    fast_span,
    slow_span,
    cost_bps=2.0,
):
    frame = pd.DataFrame({"close": close}).copy()

    frame["fast_ma"] = (
        frame["close"]
        .ewm(
            span=fast_span,
            adjust=False,
            min_periods=fast_span,
        )
        .mean()
    )

    frame["slow_ma"] = (
        frame["close"]
        .ewm(
            span=slow_span,
            adjust=False,
            min_periods=slow_span,
        )
        .mean()
    )

    ready = frame[["fast_ma", "slow_ma"]].notna().all(axis=1)

    frame["signal"] = np.where(
        ready & frame["fast_ma"].gt(frame["slow_ma"]),
        1.0,
        0.0,
    )

    frame["position"] = (
        frame["signal"]
        .shift(1)
        .fillna(0.0)
    )

    frame["turnover"] = (
        frame["position"]
        .diff()
        .fillna(frame["position"])
        .abs()
    )

    frame["asset_return"] = (
        frame["close"]
        .pct_change(fill_method=None)
        .fillna(0.0)
    )

    frame["gross_return"] = (
        frame["position"]
        * frame["asset_return"]
    )

    frame["net_return"] = (
        frame["gross_return"]
        - frame["turnover"] * cost_bps / 10_000
    )

    return frame


sma_20_100 = run_sma_crossover(
    prices["close"],
    20,
    100,
    COST_BPS,
)

ewma_20_100 = run_ewma_crossover(
    prices["close"],
    20,
    100,
    COST_BPS,
)

compare_start = max(
    sma_20_100["slow_ma"].first_valid_index(),
    ewma_20_100["slow_ma"].first_valid_index(),
)

sma_compare = sma_20_100.loc[compare_start:]
ewma_compare = ewma_20_100.loc[compare_start:]

comparison_metrics = pd.DataFrame(
    {
        "20/100 SMA": summary_from_returns(
            sma_compare["net_return"],
            periods_per_year=252,
            position=sma_compare["position"],
            turnover=sma_compare["turnover"],
        ),
        "20/100 EWMA": summary_from_returns(
            ewma_compare["net_return"],
            periods_per_year=252,
            position=ewma_compare["position"],
            turnover=ewma_compare["turnover"],
        ),
    }
)

display(comparison_metrics.round(4))

signal_comparison = pd.DataFrame(
    {
        "SMA position": sma_compare["position"],
        "EWMA position": ewma_compare["position"],
    }
).dropna()

disagreement = (
    signal_comparison["SMA position"]
    .ne(signal_comparison["EWMA position"])
    .mean()
)

print(
    f"SMA and EWMA positions differ on "
    f"{disagreement:.1%} of comparable trading days."
)


,20/100 SMA,20/100 EWMA
Total return,0.8304,0.6593
Annualized return,0.1110,0.0922
Annualized volatility,0.1173,0.1157
Sharpe ratio,0.9567,0.8203
Maximum drawdown,-0.1928,-0.2601
Entries,7.0000,9.0000
Time in market,0.7892,0.8010
Turnover,13.0000,17.0000


SMA and EWMA positions differ on 4.8% of comparable trading days.


# Part II — Cross-sectional momentum

The moving-average strategy above is time-series momentum: SPY is compared with its own price history.

Cross-sectional momentum compares several assets at the same date. The assets are ranked by a momentum score, then the portfolio is long the strongest group and short the weakest group.

For the capstone, use the eleven U.S. sector ETFs:

`XLB, XLC, XLE, XLF, XLI, XLK, XLP, XLRE, XLU, XLV, XLY`.

The portfolio will be equal-weighted, dollar-neutral, and rebalanced monthly.


## 9. Construct a cross-sectional momentum score

A common monthly momentum signal uses the previous twelve months while excluding the most recent month.

Using monthly closing prices,

$$
M_{i,t}
=
\frac{P_{i,t-1}}{P_{i,t-12}}-1.
$$

The one-month skip reduces the direct effect of very recent price movement on the ranking.

The score is relative. A bottom-ranked sector can have a positive momentum score if the other sectors performed better.

### Check

If every sector has a positive momentum score, can the strategy still hold short positions?

<details>
<summary>Solution</summary>

Yes. Cross-sectional momentum ranks the assets relative to one another. The bottom-ranked sectors are short even if their individual momentum scores are positive.

</details>


In [66]:
SECTOR_ETFS = [
    "XLB",
    "XLC",
    "XLE",
    "XLF",
    "XLI",
    "XLK",
    "XLP",
    "XLRE",
    "XLU",
    "XLV",
    "XLY",
]

sector_series = {}

for symbol in SECTOR_ETFS:
    frame = fetch_bars(
        symbol,
        TimeFrame.Day,
        DAILY_START,
        DAILY_END,
    )
    sector_series[symbol] = frame["close"].rename(symbol)

sector_close = pd.concat(
    sector_series.values(),
    axis=1,
    join="inner",
).sort_index()

sector_close = sector_close.dropna()

print(
    f"Sector sample: {sector_close.index.min().date()} "
    f"to {sector_close.index.max().date()}"
)

display(sector_close.head())


Sector sample: 2020-07-27 to 2026-09-21


,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
timestamp,,,,,,,,,,,
2020-07-27 04:00:00+00:00,27.59,53.80,14.79,21.56,65.78,50.99,52.92,28.48,24.62,95.77,64.86
2020-07-28 04:00:00+00:00,27.01,53.26,14.53,21.49,65.36,50.40,53.13,29.10,25.00,95.70,64.19
2020-07-29 04:00:00+00:00,27.21,53.73,14.85,21.94,66.30,51.11,53.22,29.63,25.11,96.68,64.85
2020-07-30 04:00:00+00:00,26.69,53.89,14.27,21.54,65.66,51.36,53.21,29.37,25.11,95.97,64.79
2020-07-31 04:00:00+00:00,26.73,54.49,14.21,21.55,65.42,52.66,53.18,29.35,25.17,95.46,65.14


In [67]:
# "ME" is the current pandas month-end alias.
# The fallback keeps the notebook compatible with older pandas versions.

try:
    monthly_close = sector_close.resample("ME").last()
except ValueError:
    monthly_close = sector_close.resample("M").last()

LOOKBACK_MONTHS = 12
SKIP_MONTHS = 1

momentum_score = (
    monthly_close.shift(SKIP_MONTHS)
    / monthly_close.shift(LOOKBACK_MONTHS)
    - 1
)

display(momentum_score.dropna().tail().round(3))


,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
timestamp,,,,,,,,,,,
2026-05-31 00:00:00+00:00,0.216,0.163,0.509,0.041,0.239,0.388,0.046,0.101,0.176,0.120,0.116
2026-06-30 00:00:00+00:00,0.183,0.077,0.360,-0.003,0.186,0.514,0.045,0.089,0.112,0.124,0.119
2026-07-31 00:00:00+00:00,0.180,0.010,0.257,0.040,0.234,0.458,0.070,0.099,0.088,0.238,0.067
2026-08-31 00:00:00+00:00,0.112,-0.016,0.359,0.071,0.198,0.343,0.082,0.101,0.081,0.203,0.010
2026-09-30 00:00:00+00:00,0.192,-0.050,0.463,0.085,0.145,0.327,0.107,0.075,-0.012,0.241,-0.021


## 10. Rank the assets and form the portfolio

At each rebalance date:

1. rank sectors from highest to lowest momentum score;
2. go long the top three;
3. go short the bottom three;
4. assign 50% total weight to the long side and 50% total weight to the short side.

Each long position therefore has weight $+1/6$, and each short position has weight $-1/6$. Net exposure is zero and gross exposure is one.

The ranking is observed at month-end. The resulting weights are applied to the following month's return.

### Exercise

Complete the sort direction for a momentum ranking.

```python
ranked = row.dropna().sort_values(ascending=____)
```

<details>
<summary>Hint</summary>

The strongest momentum score should appear first.

</details>

<details>
<summary>Solution</summary>

```python
ranked = row.dropna().sort_values(ascending=False)
```

</details>


In [68]:
# Exercise cell.

# row = momentum_score.dropna().iloc[-1]
# ranked = row.dropna().sort_values(ascending=____)
# ranked


In [69]:
N_LONG = 3
N_SHORT = 3


def make_cross_sectional_weights(
    score_row,
    n_long=3,
    n_short=3,
):
    ranked = (
        score_row
        .dropna()
        .sort_values(ascending=False)
    )

    weights = pd.Series(
        0.0,
        index=score_row.index,
        dtype=float,
    )

    if len(ranked) < n_long + n_short:
        return weights

    long_assets = ranked.index[:n_long]
    short_assets = ranked.index[-n_short:]

    weights.loc[long_assets] = 0.5 / n_long
    weights.loc[short_assets] = -0.5 / n_short

    return weights


target_weights = momentum_score.apply(
    make_cross_sectional_weights,
    axis=1,
    n_long=N_LONG,
    n_short=N_SHORT,
)

latest_date = target_weights.dropna(how="all").index[-1]

latest_ranking = pd.DataFrame(
    {
        "momentum_score": momentum_score.loc[latest_date],
        "target_weight": target_weights.loc[latest_date],
    }
).sort_values("momentum_score", ascending=False)

latest_ranking["rank"] = (
    latest_ranking["momentum_score"]
    .rank(ascending=False, method="first")
    .astype("Int64")
)

display(
    latest_ranking[
        ["rank", "momentum_score", "target_weight"]
    ].round(3)
)


,rank,momentum_score,target_weight
XLE,1,0.463,0.167
XLK,2,0.327,0.167
XLV,3,0.241,0.167
XLB,4,0.192,0.000
XLI,5,0.145,0.000
XLP,6,0.107,0.000
XLF,7,0.085,0.000
XLRE,8,0.075,0.000
XLU,9,-0.012,-0.167
XLY,10,-0.021,-0.167


## 11. Backtest the cross-sectional portfolio

Monthly asset returns are

$$
r_{i,t}
=
\frac{P_{i,t}}{P_{i,t-1}}-1.
$$

The portfolio uses the previous month's target weights:

$$
r^{p}_t
=
\sum_i w_{i,t-1}r_{i,t}.
$$

Transaction cost is based on the absolute change in portfolio weights.


In [70]:
monthly_returns = (
    monthly_close
    .pct_change(fill_method=None)
)

held_weights = (
    target_weights
    .shift(1)
    .fillna(0.0)
)

weight_change = (
    held_weights
    .diff()
)

if len(weight_change) > 0:
    weight_change.iloc[0] = held_weights.iloc[0]

monthly_turnover = (
    weight_change
    .abs()
    .sum(axis=1)
)

cross_gross_return = (
    held_weights
    * monthly_returns
).sum(axis=1, min_count=1)

cross_net_return = (
    cross_gross_return
    - monthly_turnover * COST_BPS / 10_000
)

cross_sectional = pd.DataFrame(
    {
        "gross_return": cross_gross_return,
        "net_return": cross_net_return,
        "turnover": monthly_turnover,
    }
).dropna(subset=["gross_return", "net_return"])

active_start = (
    held_weights.abs().sum(axis=1).gt(0)
)

if active_start.any():
    first_active = active_start[active_start].index[0]
    cross_sectional = cross_sectional.loc[first_active:]
    held_weights = held_weights.loc[first_active:]

cross_metrics = pd.DataFrame(
    {
        "Cross-sectional gross": summary_from_returns(
            cross_sectional["gross_return"],
            periods_per_year=12,
            turnover=cross_sectional["turnover"],
        ),
        "Cross-sectional net": summary_from_returns(
            cross_sectional["net_return"],
            periods_per_year=12,
            turnover=cross_sectional["turnover"],
        ),
    }
)

display(cross_metrics.round(4))

cross_equity = pd.DataFrame(
    {
        "Gross": (1 + cross_sectional["gross_return"]).cumprod(),
        "Net": (1 + cross_sectional["net_return"]).cumprod(),
    }
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=cross_equity.index,
        y=cross_equity["Gross"],
        name="Gross",
        line={"color": ACCENT, "width": 2.2},
    )
)

fig.add_trace(
    go.Scatter(
        x=cross_equity.index,
        y=cross_equity["Net"],
        name="Net",
        line={"color": SECONDARY_ACCENT, "width": 2.2},
    )
)

style_chart(
    fig,
    title="Cross-sectional sector momentum",
    source_text=(
        f"12-1 momentum · top {N_LONG} / bottom {N_SHORT} · "
        f"{COST_BPS:.0f} bps"
    ),
)
label_axes(fig, x_title="Date", y_title="Growth of $1")
fig.show()



,Cross-sectional gross,Cross-sectional net
Total return,0.1678,0.1616
Annualized return,0.0305,0.0294
Annualized volatility,0.0697,0.0697
Sharpe ratio,0.4657,0.4509
Maximum drawdown,-0.1143,-0.1158
Turnover,26.6667,26.6667


## 12. Training and test comparison

The same chronological split can be applied to the completed long-short portfolio. No cross-sectional parameters are selected here; the split is used only to compare the two periods.

A later version could test the lookback horizon, skip period, number of long positions, and number of short positions using the same sensitivity process used for the SMA strategy.


In [72]:
cross_split_row = int(len(cross_sectional) * 0.70)
cross_split_date = cross_sectional.index[cross_split_row]

cross_train = cross_sectional.loc[
    cross_sectional.index < cross_split_date
]

cross_test = cross_sectional.loc[
    cross_sectional.index >= cross_split_date
]

cross_split_metrics = pd.DataFrame(
    {
        "Training": summary_from_returns(
            cross_train["net_return"],
            periods_per_year=12,
            turnover=cross_train["turnover"],
        ),
        "Test": summary_from_returns(
            cross_test["net_return"],
            periods_per_year=12,
            turnover=cross_test["turnover"],
        ),
    }
)

print(f"Cross-sectional test begins: {cross_split_date.date()}")

percent_metrics = [
    "Total return",
    "Annualized return",
    "Annualized volatility",
    "Maximum drawdown",
]

cross_split_display = cross_split_metrics.style.format(
    "{:.2%}",
    subset=pd.IndexSlice[percent_metrics, :],
).format(
    "{:.2f}",
    subset=pd.IndexSlice[["Sharpe ratio", "Turnover"], :],
)

display(cross_split_display)


Cross-sectional test begins: 2025-03-31


,Training,Test
Total return,7.80%,7.75%
Annualized return,2.12%,4.83%
Annualized volatility,7.12%,6.79%
Sharpe ratio,0.33,0.73
Maximum drawdown,-11.58%,-6.68%
Turnover,19.33,7.33


## 14. Final comparison: strategies versus SPY

This final view compares every completed strategy with SPY buy-and-hold over the same months in which the cross-sectional portfolio is active. It is a fair comparison because no strategy receives extra months of performance.

Use the table and chart together:

- Higher annualized return and Sharpe ratio are better, holding risk in mind
- Lower volatility and a smaller maximum drawdown mean a smoother path
- Correlation and beta near zero indicate returns less dependent on the broad market
- A dollar-neutral cross-sectional strategy does not need to beat SPY in a bull market to be useful; its goal is a distinct return stream with reasonable risk-adjusted performance after costs


In [73]:
# Put SPY, 20/100 SMA, 20/100 EWMA, and cross-sectional momentum on one monthly sample.
try:
    resample_rule = "ME"
    spy_monthly_close = prices["close"].resample(resample_rule).last()
except ValueError:
    resample_rule = "M"
    spy_monthly_close = prices["close"].resample(resample_rule).last()

sma_monthly_return = (
    (1 + sma_20_100["net_return"]).resample(resample_rule).prod() - 1
)
ewma_monthly_return = (
    (1 + ewma_20_100["net_return"]).resample(resample_rule).prod() - 1
)

comparison_returns = pd.DataFrame(
    {
        "Cross-sectional net": cross_sectional["net_return"],
        "20/100 SMA net": sma_monthly_return,
        "20/100 EWMA net": ewma_monthly_return,
        "SPY buy and hold": spy_monthly_close.pct_change(
            fill_method=None
        ),
    }
).dropna()

comparison_turnover = pd.DataFrame(
    {
        "Cross-sectional net": cross_sectional["turnover"],
        "20/100 SMA net": sma_20_100["turnover"].resample(
            resample_rule
        ).sum(),
        "20/100 EWMA net": ewma_20_100["turnover"].resample(
            resample_rule
        ).sum(),
    }
).reindex(comparison_returns.index)

comparison_metrics = pd.DataFrame(
    {
        name: summary_from_returns(
            comparison_returns[name],
            periods_per_year=12,
            turnover=(
                comparison_turnover[name]
                if name in comparison_turnover
                else None
            ),
        )
        for name in comparison_returns.columns
    }
)

spy_returns = comparison_returns["SPY buy and hold"]
market_exposure = pd.DataFrame(
    {
        "Correlation to SPY": comparison_returns.corrwith(spy_returns),
        "Beta to SPY": comparison_returns.apply(
            lambda returns: returns.cov(spy_returns) / spy_returns.var()
        ),
    }
).T

percent_rows = [
    "Total return",
    "Annualized return",
    "Annualized volatility",
    "Maximum drawdown",
]

comparison_display = comparison_metrics.style.format(
    "{:.2%}",
    subset=pd.IndexSlice[percent_rows, :],
).format(
    "{:.2f}",
    subset=pd.IndexSlice[["Sharpe ratio", "Turnover"], :],
)

print(f"Common comparison sample: {len(comparison_returns)} months")
display(comparison_display)
display(market_exposure.style.format("{:.2f}"))

comparison_equity = (1 + comparison_returns).cumprod()

fig = go.Figure()

for name, color in zip(
    comparison_equity.columns,
    [SECONDARY_ACCENT, ACCENT, "#6B4C9A", PRICE],
):
    fig.add_trace(
        go.Scatter(
            x=comparison_equity.index,
            y=comparison_equity[name],
            name=name,
            line={"color": color, "width": 2.2},
        )
    )

style_chart(
    fig,
    title="Momentum strategies versus SPY",
    source_text="Net returns · common monthly sample",
)
label_axes(fig, x_title="Date", y_title="Growth of $1")
fig.show()


Common comparison sample: 62 months


,Cross-sectional net,20/100 SMA net,20/100 EWMA net,SPY buy and hold
Total return,16.16%,52.65%,38.38%,89.23%
Annualized return,2.94%,8.53%,6.49%,13.14%
Annualized volatility,6.97%,11.52%,11.88%,15.63%
Sharpe ratio,0.45,0.77,0.59,0.87
Maximum drawdown,-11.58%,-17.55%,-23.45%,-23.91%
Turnover,26.67,12.00,16.00,nan


,Cross-sectional net,20/100 SMA net,20/100 EWMA net,SPY buy and hold
Correlation to SPY,-0.03,0.72,0.74,1.00
Beta to SPY,-0.01,0.53,0.56,1.00


### Extension

Choose one part of the strategy to modify. You may use ChatGPT, Codex, or another coding tool to help implement it.

**Your submission should include:**

- the change you tested and why you chose it;
- the code used to implement it;
- a comparison with the original strategy;
- either a parameter-sensitivity analysis or train/test comparison;
- 2–3 sentences describing what you found.

**Possible directions include:**

- EWMA parameter sensitivity;
- signal-strength thresholds;
- volatility scaling;
- transaction-cost sensitivity;
- alternative cross-sectional lookbacks;
- different numbers of long and short positions;
- different rebalance frequencies;
- another momentum signal.